# 3_14 Train multi-class sequence classifier (GRU)

Lean runnable training notebook (no slide deck).

Classify short noisy 2D trajectories into **circle / zigzag / drift**
with a tiny GRU + linear head and cross-entropy.

| Piece | Role |
|-------|------|
| data | length-`T` paths in `R^2` |
| model | `GRU(2→16)` then `Linear(16→3)` on final hidden state |
| loss | `F.cross_entropy` |
| loop | Adam + mini-batches; monitor val; report test once |


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(46)
np.random.seed(46)

CLASS_NAMES = ["circle", "zigzag", "drift"]
T, N_PER, NOISE = 20, 80, 0.18
EPOCHS, LR, BATCH_SIZE = 40, 0.02, 32


def make_circle(rng):
    t = np.linspace(0, 2 * np.pi, T, endpoint=False)
    r = 0.85 + 0.15 * rng.randn()
    phase = rng.uniform(0, 2 * np.pi)
    xy = np.stack([r * np.cos(t + phase), r * np.sin(t + phase)], axis=1)
    return xy + NOISE * rng.randn(T, 2)


def make_zigzag(rng):
    t = np.linspace(-1.0, 1.0, T)
    amp = 0.75 + 0.15 * rng.randn()
    y = amp * ((np.arange(T) % 2) * 2 - 1)
    xy = np.stack([t, y], axis=1)
    return xy + NOISE * rng.randn(T, 2)


def make_drift(rng):
    t = np.linspace(-1.0, 1.0, T)
    a = 0.55 + 0.25 * rng.randn()
    xy = np.stack([t, a * t], axis=1)
    return xy + NOISE * rng.randn(T, 2)


MAKERS = [make_circle, make_zigzag, make_drift]
seqs, labels = [], []
for c, maker in enumerate(MAKERS):
    for i in range(N_PER):
        rng = np.random.RandomState(1000 * c + i)
        seqs.append(maker(rng).astype(np.float32))
        labels.append(c)

X = torch.from_numpy(np.stack(seqs, axis=0))  # (N, T, 2)
y = torch.tensor(labels, dtype=torch.long)
perm = torch.randperm(len(y))
X, y = X[perm], y[perm]
print(X.shape, y.shape, CLASS_NAMES)


## Model

Read the path step-by-step; classify from the **final** GRU hidden state.


In [ ]:
class SeqClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.GRU(2, 16, batch_first=True)
        self.fc = nn.Linear(16, 3)

    def forward(self, x):          # x: (B, T, 2)
        _, h = self.rnn(x)         # h: (1, B, 16)
        return self.fc(h.squeeze(0))


def dramatic_init_(model: SeqClassifier) -> None:
    """Recoverable wrong head so early epochs look clearly bad."""
    with torch.no_grad():
        W = model.fc.weight.detach().clone()
        model.fc.weight.copy_(torch.roll(W, shifts=1, dims=0))
        model.fc.bias.copy_(torch.tensor([0.8, -0.4, -0.4]))


n = len(y)
n_train, n_val = int(0.60 * n), int(0.20 * n)
X_tr, y_tr = X[:n_train], y[:n_train]
X_va, y_va = X[n_train:n_train + n_val], y[n_train:n_train + n_val]
X_te, y_te = X[n_train + n_val:], y[n_train + n_val:]

train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_va, y_va), batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(TensorDataset(X_te, y_te), batch_size=BATCH_SIZE, shuffle=False)

torch.manual_seed(7)
model = SeqClassifier()
dramatic_init_(model)
opt = torch.optim.Adam(model.parameters(), lr=LR)
print(model)


## Training loop


In [ ]:
def run_epoch(loader, train=True):
    model.train(train)
    total, n_ex, correct = 0.0, 0, 0
    for xb, yb in loader:              # xb: (B, T, 2)
        logits = model(xb)
        loss = F.cross_entropy(logits, yb)
        if train:
            opt.zero_grad()
            loss.backward()
            opt.step()
        total += loss.item() * len(yb)
        n_ex += len(yb)
        correct += (logits.argmax(dim=1) == yb).sum().item()
    return total / n_ex, correct / n_ex


for epoch in range(EPOCHS):
    tr, tr_acc = run_epoch(train_loader, train=True)
    with torch.no_grad():
        va, va_acc = run_epoch(val_loader, train=False)
    if epoch == 0 or (epoch + 1) % 10 == 0:
        print(
            f"epoch {epoch+1:02d}/{EPOCHS}  "
            f"train={tr:.3f}({tr_acc:.3f})  val={va:.3f}({va_acc:.3f})"
        )


## Inference + one-shot test report


In [ ]:
model.eval()

def predict(seq):
    # seq: (T, 2) or (B, T, 2)
    x = seq.reshape(-1, seq.shape[-2], 2)
    with torch.no_grad():
        logits = model(x)
        prob = torch.softmax(logits, dim=1)
        pred = prob.argmax(dim=1)
    return pred, prob


pred, prob = predict(X_te[0:1])
print(CLASS_NAMES[int(pred)], prob.numpy().round(3))

te_loss, te_acc = run_epoch(test_loader, train=False)
print(f"test CE={te_loss:.3f}  acc={te_acc:.3f}")
# Do not tune LR / epochs on this number.
